In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_absolute_percentage_error

In [2]:
data = pd.read_csv('cpi_data.csv')
data['Date'] = pd.to_datetime(data['Date'])
data = data.set_index('Date').asfreq('MS')

In [3]:
categories = data.columns

In [ ]:


for cat in categories:
    result = adfuller(data[cat].dropna())
    
    p_value = result[1]
    if p_value < 0.05:
        status = "Stationary"
    else:
        status = "Non-Stationary"
        
    
    
    print(f"{cat:15} | p-value: {p_value:.4f} | Status: {status:15} ")

Education       | p-value: 0.9620 | Status: Non-Stationary  
Energy          | p-value: 0.8168 | Status: Non-Stationary  
Food            | p-value: 0.9450 | Status: Non-Stationary  
Housing         | p-value: 0.9352 | Status: Non-Stationary  
Medical         | p-value: 0.9803 | Status: Non-Stationary  
Transportation  | p-value: 0.9282 | Status: Non-Stationary  


In [5]:
date = '2026-01-01'

In [ ]:
import itertools
import warnings
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_percentage_error

warnings.filterwarnings("ignore")

p = q = range(0, 2) 
d = [1]
D = [1]
s = [12]
pdq = list(itertools.product(p, d, q))
seasonal_pdq = [(x[0], x[1], x[2], x[3]) for x in list(itertools.product(p, D, q, s))]

clean_best_params = {}
mape_results = {}

print("--- LEAK-PROOF SARIMAX WORKFLOW ---\n")

for cat in categories:
    print(f"Processing {cat}...")
    
    train_data = data[cat][:date]
    test_data = data[cat][date:]
    
    lowest_aic = float('inf')
    best_combo = None
    
    for param in pdq:
        for param_seasonal in seasonal_pdq:
            try:
                model = SARIMAX(train_data, 
                                order=param,
                                seasonal_order=param_seasonal,
                                enforce_stationarity=False,
                                enforce_invertibility=False)
                results = model.fit(disp=False)
                
                if results.aic < lowest_aic:
                    lowest_aic = results.aic
                    best_combo = (param, param_seasonal)
            except:
                continue
                
    clean_best_params[cat] = best_combo
    
    order = best_combo[0]
    seasonal_order = best_combo[1]
    
    final_train_model = SARIMAX(train_data,
                                order=order,
                                seasonal_order=seasonal_order,
                                enforce_stationarity=True,
                                enforce_invertibility=True)
    final_train_results = final_train_model.fit(disp=False)
    
    predictions = final_train_results.forecast(steps=len(test_data))
    
    mape = mean_absolute_percentage_error(test_data, predictions) * 100
    mape_results[cat] = mape
    
    print(f"  Best Params: SARIMAX{order}{seasonal_order} | MAPE: {mape:.2f}%\n")

--- LEAK-PROOF SARIMAX WORKFLOW ---

Processing Education...
  Best Params: SARIMAX(1, 1, 1)(0, 1, 1, 12) | MAPE: 0.32%

Processing Energy...
  Best Params: SARIMAX(1, 1, 1)(0, 1, 1, 12) | MAPE: 7.98%

Processing Food...
  Best Params: SARIMAX(1, 1, 1)(0, 1, 1, 12) | MAPE: 0.27%

Processing Housing...
  Best Params: SARIMAX(1, 1, 1)(0, 1, 1, 12) | MAPE: 0.22%

Processing Medical...
  Best Params: SARIMAX(0, 1, 1)(1, 1, 1, 12) | MAPE: 0.14%

Processing Transportation...
  Best Params: SARIMAX(0, 1, 1)(0, 1, 1, 12) | MAPE: 3.31%



In [7]:
print("="*50)
print("FINAL LEAK-PROOF MAPE SCORES:")
for cat, mape in mape_results.items():
    print(f"{cat:15} : {mape:.2f}%")

FINAL LEAK-PROOF MAPE SCORES:
Education       : 0.32%
Energy          : 7.98%
Food            : 0.27%
Housing         : 0.22%
Medical         : 0.14%
Transportation  : 3.31%


In [ ]:
from dateutil.relativedelta import relativedelta

print("\n--- FORECASTING NEXT 6 MONTHS ---")
last_date = data.index[-1]
future_dates = [last_date + relativedelta(months=i) for i in range(1, 7)]
future_forecasts = {}

for cat in categories:
    order = clean_best_params[cat][0]
    seasonal_order = clean_best_params[cat][1]
    
    model_final = SARIMAX(data[cat], 
                          order=order, 
                          seasonal_order=seasonal_order,
                          enforce_stationarity=True,
                          enforce_invertibility=True)
    results_final = model_final.fit(disp=False)
    
    forecast = results_final.forecast(steps=6)
    future_forecasts[cat] = forecast.values

forecast_df = pd.DataFrame(future_forecasts)
forecast_df.index = future_dates
forecast_df.index.name = 'Date'

print("="*60)
print("FINAL 6-MONTH CPI FORECAST (Leak-Proof SARIMAX)")
print("="*60)
print(forecast_df.to_string())


--- FORECASTING NEXT 6 MONTHS ---
FINAL 6-MONTH CPI FORECAST (Leak-Proof SARIMAX)
             Education      Energy        Food     Housing     Medical  Transportation
Date                                                                                  
2026-06-01  147.811711  355.675262  349.787518  360.246999  594.058979      303.184978
2026-07-01  148.058339  353.091769  350.579542  361.354322  594.529135      303.146195
2026-08-01  148.537257  351.768592  351.173156  362.483885  596.067206      302.263530
2026-09-01  148.922907  351.330972  352.077451  363.592279  596.599959      301.415298
2026-10-01  148.882944  349.930003  352.893070  364.319023  597.636360      301.972629
2026-11-01  148.790618  346.281285  352.708063  364.987243  598.151331      301.045394


In [10]:
forecast_df.to_csv('sarimax_predictions.csv')